# 🧠 Tripolar EEG — Group Analysis (All Subjects)

This notebook loads **all subjects**, runs the per-subject analysis, and produces **group-level statistical comparisons** of tEEG vs conventional electrodes.

**Prerequisites:** `eeg_analysis.py` and all subject data files in `data/`.

---

### Research Question
Are tripolar tEEG electrodes as effective as conventional disc electrodes for detecting:
1. **Visually-induced alpha waves** (checkerboard VEP paradigm)
2. **Spontaneous alpha** (eyes-open vs eyes-closed, Berger effect)


In [ ]:
import sys
sys.path.insert(0, '.')
from eeg_analysis import *
from scipy import stats

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
print("Ready ✓")

## 1. Discover & Load All Subjects

In [ ]:
DATA_DIR = 'data/'

subjects_info = discover_subjects(DATA_DIR)
print(f"Found {len(subjects_info)} subjects:\n")
for s in subjects_info:
    avg_str = "✓ avg" if s['avg'] else "✗ avg"
    vmrk_str = "✓ mrk" if s['vmrk'] else "✗ mrk"
    print(f"  {s['name']:<15s}  {vmrk_str}  {avg_str}  ({s['basename']})")

In [ ]:
# Load and analyze all subjects
all_subjects = {}
all_results = {}
failed = []

for info in subjects_info:
    name = info['name']
    try:
        print(f"Loading {name}...", end=" ")
        subj = load_subject(DATA_DIR, subject_info=info)
        print(f"({subj['n_samples']/FS:.0f}s, {len(subj['stim_samples'])} stim, "
              f"{len(subj['events_oc'])} oc markers)", end=" ")
        
        res = analyze_subject(subj)
        all_subjects[name] = subj
        all_results[name] = res
        print("✓")
    except Exception as e:
        print(f"✗ FAILED: {e}")
        failed.append((name, str(e)))

print(f"\n{'═'*50}")
print(f"Successfully loaded: {len(all_results)} / {len(subjects_info)} subjects")
if failed:
    print(f"Failed: {[f[0] for f in failed]}")

## 2. Per-Subject Summary Dashboards

In [ ]:
for name in all_results:
    plot_subject_summary(all_subjects[name], all_results[name])

## 3. Compile Group Metrics

In [ ]:
group = get_group_metrics(all_results)
n_subj = len(group['subject_names'])

print(f"Group metrics compiled for {n_subj} subjects:")
print(f"  Subjects: {group['subject_names']}")
print(f"  Each metric shape: ({n_subj}, {N_CHANNELS})")

# Electrode type groupings
type_names = ['SW Conv', 'SW tEEG', 'Paste Conv', 'Paste tEEG', 'Disc']
type_indices = [IDX_SW_CONV, IDX_SW_TEEG, IDX_PASTE_CONV, IDX_PASTE_TEEG, IDX_DISC]
type_colors = ['#95a5a6', '#3498db', '#9b59b6', '#2ecc71', '#e74c3c']

## 4. Group Comparison — Box Plots

Each box shows the distribution across all subjects for that electrode type. If tEEG boxes overlap with the Disc box, that's evidence they're comparable.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f'Group Comparison Across {n_subj} Subjects', fontsize=15, fontweight='bold')

metrics_to_plot = [
    ('alpha_snr', 'Alpha SNR (dB)', None),
    ('alpha_reactivity', 'Alpha Reactivity (Closed/Open)', 1.0),
    ('disc_correlation', 'Correlation with Disc (Ch11)', None),
    ('vep_p2p', 'VEP Peak-to-Peak (µV)', None),
]

for ax, (key, title, hline) in zip(axes.flatten(), metrics_to_plot):
    data_by_type = []
    for idxs in type_indices:
        # Average across channels within each type, then collect per subject
        vals = np.nanmean(group[key][:, idxs], axis=1)  # (n_subj,)
        data_by_type.append(vals)
    
    bp = ax.boxplot(data_by_type, labels=type_names, patch_artist=True,
                     widths=0.6, showmeans=True,
                     meanprops=dict(marker='D', markerfacecolor='white', markersize=6))
    for patch, color in zip(bp['boxes'], type_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    # Overlay individual subject points
    for j, vals in enumerate(data_by_type):
        x_jitter = np.random.normal(j + 1, 0.08, size=len(vals))
        ax.scatter(x_jitter, vals, color=type_colors[j], s=25, alpha=0.8,
                   edgecolors='black', linewidths=0.5, zorder=3)
    
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.tick_params(labelsize=9)
    if hline is not None:
        ax.axhline(hline, color='gray', ls='--', lw=0.8)

plt.tight_layout(); plt.show()

## 5. Statistical Tests

**Paired comparisons** (within-subject): For each subject, we average across channels of the same type and compare SW tEEG vs Disc using a paired t-test (or Wilcoxon signed-rank if n < 20).

In [ ]:
print(f"Paired comparisons: SW tEEG vs Disc (n={n_subj} subjects)")
print(f"Test: {'Paired t-test' if n_subj >= 20 else 'Wilcoxon signed-rank (small n)'}")
print("═" * 75)
print(f"{'Metric':<30} {'SW tEEG':>10} {'Disc':>10} {'p-value':>10} {'Sig?':>6}")
print("─" * 75)

for key, label, _ in metrics_to_plot:
    teeg_vals = np.nanmean(group[key][:, IDX_SW_TEEG], axis=1)
    disc_vals = np.nanmean(group[key][:, IDX_DISC], axis=1)
    
    # Remove NaN pairs
    valid = ~(np.isnan(teeg_vals) | np.isnan(disc_vals))
    tv, dv = teeg_vals[valid], disc_vals[valid]
    
    if len(tv) >= 3:
        if len(tv) >= 20:
            stat, p = stats.ttest_rel(tv, dv)
        else:
            stat, p = stats.wilcoxon(tv, dv)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        print(f"{label:<30} {np.mean(tv):>10.3f} {np.mean(dv):>10.3f} {p:>10.4f} {sig:>6}")
    else:
        print(f"{label:<30} {'insufficient data':>30}")

print()
print("Also comparing: Paste tEEG vs Disc")
print("─" * 75)

for key, label, _ in metrics_to_plot:
    paste_vals = np.nanmean(group[key][:, IDX_PASTE_TEEG], axis=1)
    disc_vals = np.nanmean(group[key][:, IDX_DISC], axis=1)
    valid = ~(np.isnan(paste_vals) | np.isnan(disc_vals))
    pv, dv = paste_vals[valid], disc_vals[valid]
    
    if len(pv) >= 3:
        if len(pv) >= 20:
            stat, p = stats.ttest_rel(pv, dv)
        else:
            stat, p = stats.wilcoxon(pv, dv)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        print(f"{label:<30} {np.mean(pv):>10.3f} {np.mean(dv):>10.3f} {p:>10.4f} {sig:>6}")
    else:
        print(f"{label:<30} {'insufficient data':>30}")

## 6. Per-Subject Breakdown Table

In [ ]:
for key, label, _ in metrics_to_plot:
    print(f"\n{'─'*80}")
    print(f" {label}")
    print(f"{'─'*80}")
    header = f"{'Subject':<15}"
    for tn in type_names:
        header += f" {tn:>10}"
    print(header)
    
    for s_idx, name in enumerate(group['subject_names']):
        row = f"{name:<15}"
        for idxs in type_indices:
            val = np.nanmean(group[key][s_idx, idxs])
            row += f" {val:>10.3f}"
        print(row)
    
    # Group mean
    row_mean = f"{'MEAN':<15}"
    for idxs in type_indices:
        val = np.nanmean(group[key][:, idxs])
        row_mean += f" {val:>10.3f}"
    print(f"{'─'*15}" + "─" * (11 * len(type_names)))
    print(row_mean)

## 7. Alpha Reactivity by Subject

Bar chart showing each subject's alpha reactivity for tEEG vs Disc. Bars above the dashed line (ratio > 1) indicate successful Berger effect detection.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

x = np.arange(n_subj)
w = 0.25

teeg_react = np.nanmean(group['alpha_reactivity'][:, IDX_SW_TEEG], axis=1)
paste_react = np.nanmean(group['alpha_reactivity'][:, IDX_PASTE_TEEG], axis=1)
disc_react = np.nanmean(group['alpha_reactivity'][:, IDX_DISC], axis=1)

ax.bar(x - w, teeg_react, w, color='#3498db', label='SW tEEG', edgecolor='k', lw=0.5)
ax.bar(x, paste_react, w, color='#2ecc71', label='Paste tEEG', edgecolor='k', lw=0.5)
ax.bar(x + w, disc_react, w, color='#e74c3c', label='Disc', edgecolor='k', lw=0.5)

ax.axhline(1, color='gray', ls='--', lw=1)
ax.set_xticks(x)
ax.set_xticklabels(group['subject_names'], fontsize=10)
ax.set_ylabel('Alpha Reactivity (Closed / Open)')
ax.set_title('Alpha Reactivity by Subject — tEEG vs Disc', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout(); plt.show()

## 8. Correlation Scatter: tEEG vs Disc

If tEEG tracks the disc well, points should cluster near the diagonal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('tEEG vs Disc — Per-Subject Scatter', fontsize=13, fontweight='bold')

for ax, (key, label, _) in zip(axes, metrics_to_plot[:3]):
    teeg_vals = np.nanmean(group[key][:, IDX_SW_TEEG], axis=1)
    disc_vals = np.nanmean(group[key][:, IDX_DISC], axis=1)
    
    ax.scatter(disc_vals, teeg_vals, s=80, color='#3498db', edgecolors='black',
               linewidths=0.5, zorder=3, label='SW tEEG')
    
    paste_vals = np.nanmean(group[key][:, IDX_PASTE_TEEG], axis=1)
    ax.scatter(disc_vals, paste_vals, s=80, color='#2ecc71', edgecolors='black',
               linewidths=0.5, zorder=3, marker='s', label='Paste tEEG')
    
    # Add diagonal
    all_vals = np.concatenate([disc_vals, teeg_vals, paste_vals])
    lim = [np.nanmin(all_vals) * 0.9, np.nanmax(all_vals) * 1.1]
    ax.plot(lim, lim, 'k--', lw=0.8, alpha=0.5)
    
    # Add subject labels
    for j, name in enumerate(group['subject_names']):
        ax.annotate(name, (disc_vals[j], teeg_vals[j]), fontsize=7, alpha=0.7,
                     xytext=(5, 5), textcoords='offset points')
    
    ax.set_xlabel(f'Disc — {label}')
    ax.set_ylabel(f'tEEG — {label}')
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

## 9. Save All Per-Subject Figures (Optional)

Uncomment and run to save detailed per-subject figures to `figures/<subject_name>/`.

In [ ]:
# Uncomment to generate and save all figures:

# import os
# for name in all_results:
#     save_dir = os.path.join('figures', name)
#     print(f"Generating figures for {name}...")
#     plot_subject_full(all_subjects[name], all_results[name],
#                       save_dir=save_dir, show=False)
#     print(f"  → Saved to {save_dir}/")
# print("Done!")

## 10. Conclusions

### Key Questions to Answer from This Data:

1. **Do tEEG electrodes detect the Berger effect?**  
   → Check if alpha reactivity > 1 for tEEG channels (Section 7)

2. **Are tEEG alpha metrics comparable to disc?**  
   → Check box plots (Section 4) and p-values (Section 5)

3. **Does electrolyte matter?**  
   → Compare saltwater tEEG vs paste tEEG

4. **Do tEEG electrodes capture VEPs?**  
   → Compare VEP peak-to-peak amplitudes

### Statistical Interpretation
- **p < 0.05:** Significant difference between tEEG and disc (they're NOT equivalent)
- **p > 0.05:** No significant difference (evidence FOR equivalence — good for your hypothesis!)
- With n=9, consider these preliminary results. A larger sample would give more power.

### Caveats
- Channel mapping (odd=Conv, even=tEEG) is inferred — verify with hardware docs
- Electrode scalp positions not controlled/documented across subjects
- No artifact rejection applied (ICA recommended for publication-quality results)
